## Introduction

In this notebook, we take the next step in the series.  We will now vectorize the ER data.  However, we will not yet be doing any form of GraphRAG.  We will, as in notebook `05`, vectorize the ER results as returned from Senzing in JSON format and use these vectors as the basis of the RAG.

**Note:** We will now use these embeddings for all subsequent notebooks.  So unlike `05`, you will need to be sure to run this notebook before doing the ERKG notebooks in `08`.

In [1]:
import json
import os
import grpc
import pandas as pd
from sentence_transformers import SentenceTransformer
import lancedb
from senzing import SzEngine, SzError
from senzing_grpc import SzAbstractFactoryGrpc
from senzing import SzEngineFlags

SENZING_HOST = os.getenv('SENZING_GRPC_HOST', 'senzing')
SENZING_PORT = os.getenv('SENZING_GRPC_PORT', '8261')

# Create gRPC channel and engine
grpc_url = f"{SENZING_HOST}:{SENZING_PORT}"
grpc_channel = grpc.insecure_channel(grpc_url)
sz_abstract_factory = SzAbstractFactoryGrpc(grpc_channel)
sz_engine = sz_abstract_factory.create_engine()

## Export All Resolved Entities

As before, we will get the entity report back from Senzing in JSON format.  This will be the data that we then convert to text like we did in `05` and then vectorize.  Note that we need the record-level JSON because that is where names, addresses, identifiers, and risk topics are stored.

In [2]:
entities = []
export_handle = sz_engine.export_json_entity_report(
    flags=SzEngineFlags.SZ_EXPORT_INCLUDE_ALL_ENTITIES | 
          SzEngineFlags.SZ_ENTITY_INCLUDE_RECORD_JSON_DATA |
          SzEngineFlags.SZ_ENTITY_INCLUDE_ALL_RELATIONS |
          SzEngineFlags.SZ_ENTITY_INCLUDE_ENTITY_NAME
)

count = 0
while True:
    try:
        entity_json = sz_engine.fetch_next(export_handle)
        if not entity_json:
            break
        
        entity = json.loads(entity_json)
        entities.append(entity)
        count += 1
        
        if count % 50 == 0:
            print(f"  Exported {count} entities...", end='\r')
    except StopIteration:
        break

sz_engine.close_export_report(export_handle)
print(f"\nExported {len(entities)} entities total")

# Check how many have relationships
with_rels = sum(1 for e in entities if e.get('RELATED_ENTITIES'))
print(f"Entities with relationships: {with_rels}")

  Exported 150 entities...
Exported 196 entities total
Entities with relationships: 189


## Load the Embedding Model

Downloads and initializes `all-MiniLM-L6-v2` from Hugging Face.  This model produces 384-dimension embeddings and is a solid choice for this task since it is fast, small enough to run comfortably in the workshop environment, and handles the mix of names, addresses, and risk terminology in our entity text reasonably well.

In [3]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded")
print(f"  Model: all-MiniLM-L6-v2")
print(f"  Embedding dimension: 384")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
  Model: all-MiniLM-L6-v2
  Embedding dimension: 384


## Create Entity Embeddings

As before, we will be creating text strings from JSON, in this case, the ER results returned by Senzing.  These will then again be stored in LanceDB.

In [4]:
entity_data = []

for entity in entities:
    entity_data_item = entity.get('RESOLVED_ENTITY', {})
    entity_id = entity_data_item.get('ENTITY_ID')
    entity_name = entity_data_item.get('ENTITY_NAME')
    records = entity_data_item.get('RECORDS', [])
    
    if not records:
        continue
    
    # Get entity info from FEATURES (v4 format)
    first_record = records[0]
    json_data = first_record.get('JSON_DATA', {})
    features = json_data.get('FEATURES', [])
    
    name = None
    record_type = 'UNKNOWN'
    for feat in features:
        if 'RECORD_TYPE' in feat:
            record_type = feat['RECORD_TYPE']
        if not name:
            name = feat.get('NAME_FULL') or feat.get('NAME_ORG') or feat.get('PRIMARY_NAME_ORG')
    
    # Fallback to top-level fields
    if not name:
        name = json_data.get('PRIMARY_NAME_FULL')
    if not name:
        for name_obj in json_data.get('NAMES', []):
            name = name_obj.get('NAME_FULL') or name_obj.get('NAME_ORG') or name_obj.get('PRIMARY_NAME_ORG')
            if name:
                break
    
    # Use ENTITY_NAME from export first, then FEATURES extraction, then fallback
    name = entity_name or name or f"Entity {entity_id}"
    
    data_sources = list(set([r.get('DATA_SOURCE') for r in records]))
    
    # Get addresses from FEATURES (v4) and fallback to ADDRESSES
    addresses = []
    for rec in records:
        rec_json = rec.get('JSON_DATA', {})
        for feat in rec_json.get('FEATURES', []):
            if feat.get('ADDR_FULL') and len(addresses) < 3:
                addresses.append(feat['ADDR_FULL'])
        for addr in rec_json.get('ADDRESSES', []):
            if addr.get('ADDR_FULL') and len(addresses) < 3:
                addresses.append(addr['ADDR_FULL'])
    
    # Get identifiers from FEATURES (v4) and fallback to IDENTIFIERS
    identifiers = []
    for rec in records:
        rec_json = rec.get('JSON_DATA', {})
        for feat in rec_json.get('FEATURES', []):
            id_type = feat.get('NATIONAL_ID_TYPE') or feat.get('OTHER_ID_TYPE')
            id_num = feat.get('NATIONAL_ID_NUMBER') or feat.get('OTHER_ID_NUMBER')
            if id_type and id_num and len(identifiers) < 3:
                identifiers.append(f"{id_type}: {id_num}")
        for id_obj in rec_json.get('IDENTIFIERS', []):
            id_type = id_obj.get('NATIONAL_ID_TYPE') or id_obj.get('OTHER_ID_TYPE')
            id_num = id_obj.get('NATIONAL_ID_NUMBER') or id_obj.get('OTHER_ID_NUMBER')
            if id_type and id_num and len(identifiers) < 3:
                identifiers.append(f"{id_type}: {id_num}")
    
    # Get risks from top-level fields (these are outside FEATURES)
    risks = []
    for rec in records:
        rec_json = rec.get('JSON_DATA', {})
        # Check top-level risk_topics field
        risk_topics = rec_json.get('risk_topics', '')
        if risk_topics:
            risks.extend(risk_topics.split(','))
        # Also check RISKS array
        for risk in rec_json.get('RISKS', []):
            topic = risk.get('TOPIC', '')
            if topic:
                risks.append(topic)
    
    # Get related entity info for richer text
    related = entity.get('RELATED_ENTITIES', [])
    
    # Create text description for embedding
    text_parts = [
        f"Name: {name}",
        f"Type: {record_type}",
        f"Data sources: {', '.join(data_sources)}",
        f"Records merged: {len(records)}"
    ]
    
    if addresses:
        text_parts.append(f"Address: {addresses[0]}")
    
    if identifiers:
        text_parts.append(f"Identifiers: {', '.join(identifiers[:2])}")
    
    if risks:
        text_parts.append(f"Risk topics: {', '.join(set(risks))}")
    
    if related:
        text_parts.append(f"Related to {len(related)} other entities")
    
    text = ". ".join(text_parts)
    
    # Create embedding
    embedding = embedding_model.encode(text).tolist()
    
    # Store data
    entity_data.append({
        'entity_id': entity_id,
        'name': name,
        'type': record_type,
        'text': text,
        'vector': embedding,
        'data_sources': ','.join(data_sources),
        'num_records': len(records),
        'addresses': '|'.join(addresses[:3]),
        'identifiers': '|'.join(identifiers[:3]),
        'risks': '|'.join(set(risks))
    })
    
    if len(entity_data) % 50 == 0:
        print(f"  Processed {len(entity_data)} entities...", end='\r')

print(f"\nCreated embeddings for {len(entity_data)} entities")

  Processed 150 entities...
Created embeddings for 196 entities


## Store Embeddings in LanceDB

Like before, we will start with a clean database and add the above embeddings to it. 

In [5]:
db = lancedb.connect('/workspace/lancedb_data')

# Drop all existing tables
for table_name in db.table_names():
    db.drop_table(table_name)
    print(f"Dropped table: {table_name}")

# Create new table
print("Creating new table...")
table = db.create_table('entities', entity_data)

print(f"Stored {len(entity_data)} entities in LanceDB")
print("\nData preparation complete!")
print("You can now use the RAG notebook to query this data.")

Dropped table: raw_records
Creating new table...
Stored 196 entities in LanceDB

Data preparation complete!
You can now use the RAG notebook to query this data.


/tmp/ipykernel_13489/2062147836.py:4: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  for table_name in db.table_names():


## Preview LanceDB Contents

Let's confirm what is being stored in LanceDB this time.

In [6]:
print("LanceDB Contents:")
print("="*70)

sample = table.to_pandas().head(10)
display_columns = ['entity_id', 'name', 'type', 'data_sources', 'num_records', 'risks']
print(sample[display_columns].to_string())

LanceDB Contents:
   entity_id                               name          type                   data_sources  num_records                                 risks
0     100001                    Abassin Badshah        PERSON  OPEN-SANCTIONS,OPEN-OWNERSHIP            3                          corp.disqual
1     100002                        LMAR GB LTD  ORGANIZATION                 OPEN-SANCTIONS            1                                      
2     100003            WANDLE HOLDINGS LIMITED  ORGANIZATION                 OPEN-SANCTIONS            1                       sanction.linked
3     100004  POLYUS GOLD INTERNATIONAL LIMITED  ORGANIZATION                 OPEN-SANCTIONS            1                       sanction.linked
4     100005          Firuza Nazimovna Kerimova        PERSON                 OPEN-SANCTIONS            1                    role.rca; sanction
5     100006                     ООО "ГРАНДЕКО"  ORGANIZATION                 OPEN-SANCTIONS            2  sanction.li

In [7]:
print("Closing connections...")

try:
    grpc_channel.close()
    print("Done")
except Exception as e:
    print(f"Note: {e}")

Closing connections...
Done
